# IKT466 – run all experiments in Colab

Runs the whole pipeline from the repository's scripts: data, splits, test images, training, evaluation, Grad-CAM and the summary.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

Checkpoints and results are saved to Google Drive, so nothing is lost if Colab disconnects. Training cells skip runs that are already finished, so after a disconnect you can just run all cells again.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Jorgenklauseen/MachineLearning-Project.git"   # the clone address, ending in .git
BRANCH = "pipeline"                              # the branch with the code you want to run
DRIVE_DIR = "/content/drive/MyDrive/ikt466"      # checkpoints and results are kept here

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again."

%cd /content
!rm -rf repo && git clone -q -b $BRANCH $REPO_URL repo
# Keep checkpoints/ and results/ on Drive: copy what the repo already has, then link the folders
!mkdir -p $DRIVE_DIR/checkpoints $DRIVE_DIR/results
!cp -rn repo/checkpoints/. $DRIVE_DIR/checkpoints/ 2>/dev/null; cp -rn repo/results/. $DRIVE_DIR/results/ 2>/dev/null; true
!rm -rf repo/checkpoints repo/results
!ln -s $DRIVE_DIR/checkpoints repo/checkpoints && ln -s $DRIVE_DIR/results repo/results
%cd /content/repo/src
# Colab already has torch, torchvision, scikit-learn, OpenCV and matplotlib. The pinned versions in
# requirements.txt are for local installs; the versions used here are saved to environment.txt.
!pip install -q scikit-image
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data, splits and test images

In [ ]:
# Same dataset copy and version as in the README (about 770 MB, takes a minute or two)
import os
if not os.path.exists("../data/oxford-iiit-pet/images"):
    # Only the pinned commit, without the repository's history
    !git clone -q --depth 1 https://github.com/ml4py/dataset-iiit-pet.git ../data/oxford-iiit-pet
    !git -C ../data/oxford-iiit-pet fetch -q --depth 1 origin ebb5f9e482d0597e7eb4a92f24c7275e83b96140
    !git -C ../data/oxford-iiit-pet checkout -q FETCH_HEAD

!python save_splits.py
# The two fills run at the same time, one per CPU core; skipped if already made in this session
!python make_background_test.py --skip-existing --preview ../results/figures/variants_preview.png & python make_background_test.py --fill noise --out ../data/background_test_noise --skip-existing --preview ../results/figures/variants_preview_noise.png & wait

In [ ]:
from IPython.display import Image, display
display(Image("../results/figures/variants_preview.png", width=600))   # look at these before training

## 2. SVM baseline
The full grid search (5-fold CV, 40 settings) takes hours on Colab's two CPU cores. If its results from an earlier run are in `results/svm_gridsearch*.csv`, the best setting is refitted directly (`--saved-grid`), which gives the same model in a few minutes. Skipped if the model is already in the Drive `checkpoints` folder.

In [ ]:
import os
for flag, name, grid in [("", "svm_hog", "svm_gridsearch.csv"),
                         ("--color", "svm_hog_color", "svm_gridsearch_color.csv")]:
    if os.path.exists(f"../checkpoints/{name}.joblib"):
        print(f"{name}: already trained, skipping")
    elif os.path.exists(f"../results/{grid}"):
        !python train_svm.py --saved-grid {flag}
    else:
        !python train_svm.py {flag}

## 3. CNNs: quick check, then 3 seeds each
The quick check trains on 64 images; the training loss should fall. Its checkpoint ends in `_debug` and is never evaluated.

In [ ]:
!python train_cnn.py --model pet_cnn --debug --epochs 3

In [ ]:
import os
for seed in [0, 1, 2]:
    for model in ["resnet50", "pet_cnn"]:
        if os.path.exists(f"../results/history/{model}_seed{seed}.json"):
            print(f"{model} seed {seed}: already trained, skipping")
            continue
        !python train_cnn.py --model {model} --seed {seed}

## 4. Evaluation on the original, animal-only and background-only images (grey and noise fill)

In [ ]:
!python evaluate_svm_background.py
!python evaluate_svm_background.py --data ../data/background_test_noise --tag _noise
!python evaluate_cnn_background.py
!python evaluate_cnn_background.py --data ../data/background_test_noise --tag _noise

## 5. Grad-CAM and the summary

In [ ]:
!python gradcam_share.py
!python summarize_results.py
!python plot_learning_curves.py
!pip freeze > ../results/environment.txt   # exact package versions, for the report and README

In [ ]:
from IPython.display import Image, display
for name in ["learning_curves", "accuracy_by_variant", "accuracy_by_variant_noise", "gradcam_examples"]:
    display(Image(f"../results/figures/{name}.png", width=700))